# VOC2012 — huấn luyện classifier trên Google Colab

Notebook riêng cho **Colab trên trình duyệt hoặc kernel Colab trong VS Code**.
Mục tiêu: ảnh + nhãn cấp ảnh → ResNet pretrained → đánh giá phân loại, chuẩn bị cho CAM.

**Cách chạy:** chọn kernel Colab có GPU rồi chạy các cell từ trên xuống. Trong VS Code:
`Select Kernel → Colab → New Colab Server`; chọn cấu hình GPU và đăng nhập Google.
Notebook không dùng `.venv` hoặc đường dẫn ổ D của máy bạn.

- Dữ liệu được giải nén tại `/content/data`, để đọc ảnh trong lúc train.
- Archive dữ liệu, checkpoint, lịch sử và kết quả được lưu trên Google Drive.
- Runtime mới có thể mất dữ liệu `/content`; notebook sẽ lấy lại archive từ Drive.
- Mỗi lượt chạy tạo thư mục kết quả riêng; bước tạo model bắt đầu lại từ pretrained.
- Mask chuẩn chỉ được xem để hiểu dữ liệu. Classifier học từ tên lớp trong XML.
- Dùng tập segmentation train/val gốc (1.464/1.449 ảnh), chưa phải tập trainaug.

Hướng dẫn kết nối: https://github.com/googlecolab/colab-vscode/wiki/User-Guide

## 1. Cấu hình

Giữ `IMAGE_SIZE = 244` như notebook gốc. Đổi `MODEL_NAME` thành `resnet50` nếu muốn
thử backbone lớn hơn; tạo lại model và optimizer bằng cách chạy các cell từ đầu.
`NUM_EPOCHS` là số epoch của một lượt chạy mới, không gồm bước train minh họa nào khác.

`DATA_SOURCE = "auto"`: ưu tiên archive đã lưu trên Drive, nếu chưa có thì tải từ mirror.
`"drive_archive"`: chỉ dùng archive bạn đã upload vào đường dẫn in ở mục kết nối Drive.

In [ ]:
MODEL_NAME = "resnet18"       # "resnet18" hoặc "resnet50"
IMAGE_SIZE = 244
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 1e-4
SEED = 42
NUM_WORKERS = 0
REQUIRE_GPU = True

DATA_SOURCE = "auto"          # "auto" hoặc "drive_archive"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS331_WSSS"

assert MODEL_NAME in {"resnet18", "resnet50"}
assert DATA_SOURCE in {"auto", "drive_archive"}
assert IMAGE_SIZE >= 32 and BATCH_SIZE > 0 and NUM_EPOCHS > 0

## 2. Kiểm tra runtime và thư viện

Sử dụng PyTorch GPU có sẵn trong Colab. Cell chỉ cài bổ sung thư viện còn thiếu,
không thay phiên bản torch/torchvision. Nếu kiểm tra GPU thất bại, chọn lại runtime GPU.

In [ ]:
import importlib.util
import subprocess
import sys

try:
    import google.colab
except ImportError as exc:
    raise RuntimeError("Hãy chọn kernel Google Colab trước khi chạy notebook này.") from exc

packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
}
missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import hashlib
import json
import math
import random
import shutil
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from torchvision.models import (
    resnet18,
    ResNet18_Weights,
    resnet50,
    ResNet50_Weights,
)
from torchvision.datasets.utils import check_integrity, download_url, extract_archive
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0])
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Thiết bị:", device)

if REQUIRE_GPU and device.type != "cuda":
    raise RuntimeError("Chưa có GPU. Hãy kết nối runtime GPU rồi chạy lại.")

if device.type == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(
        "GPU:", gpu.name,
        "| VRAM:", round(gpu.total_memory / 1024**3, 1), "GB",
    )

## 3. Kết nối Drive và tạo thư mục kết quả

Trong VS Code, có thể dùng `Colab: Mount Google Drive to Server...` nếu được yêu cầu
đăng nhập qua giao diện extension. Đường dẫn mount sử dụng bên dưới là `/content/drive`.
Cell sẽ in đường dẫn archive để bạn upload vào Drive khi chọn `drive_archive`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive chưa được mount đúng tại /content/drive.")

project_dir = Path(DRIVE_PROJECT_DIR)
archive_dir = project_dir / "archives"
archive_dir.mkdir(parents=True, exist_ok=True)
archive_name = "VOCtrainval_11-May-2012.tar"
drive_archive = archive_dir / archive_name

run_id = datetime.now(ZoneInfo("Asia/Bangkok")).strftime("%Y%m%d_%H%M%S_%f")
run_dir = project_dir / "runs" / f"{MODEL_NAME}_{run_id}"
run_dir.mkdir(parents=True, exist_ok=False)

print("Archive trên Drive:", drive_archive)
print("Kết quả lượt chạy:", run_dir)

## 4. Chuẩn bị VOC2012 trên ổ đĩa Colab

Lần đầu tải archive khoảng 2 GB, kiểm tra MD5 rồi giải nén. Archive được lưu lại trên
Drive để dùng cho runtime sau. Các lần sau copy archive từ Drive và giải nén vào `/content`.
Ảnh được đọc từ ổ đĩa runtime khi train; checkpoint được ghi về Drive.

Mirror: https://pjreddie.com/projects/pascal-voc-dataset-mirror/

In [ ]:
data_root = Path("/content/data")
data_root.mkdir(parents=True, exist_ok=True)
voc_root = data_root / "VOCdevkit" / "VOC2012"
local_archive = data_root / archive_name
archive_md5 = "6cd6e144f989b92b3379bac3b3de84fd"

if not voc_root.is_dir():
    if not check_integrity(local_archive, archive_md5):
        if drive_archive.is_file():
            print("Copy archive từ Drive vào runtime...")
            shutil.copy2(drive_archive, local_archive)

        elif DATA_SOURCE == "drive_archive":
            raise FileNotFoundError(f"Hãy upload archive vào: {drive_archive}")
        else:
            urls = [
                "https://data.pjreddie.com/files/" + archive_name,
                "https://thor.robots.ox.ac.uk/pascal/VOC/voc2012/" + archive_name,
            ]

            for url in urls:
                try:
                    download_url(
                        url,
                        str(data_root),
                        filename=archive_name,
                        md5=archive_md5,
                    )
                    break
                except Exception as exc:
                    print(f"Nguồn tải chưa thành công: {url} ({exc})")
            else:
                raise RuntimeError("Tải thất bại. Có thể upload archive lên Drive và chạy lại.")

    if not check_integrity(local_archive, archive_md5):
        raise RuntimeError("Archive không đúng MD5. Hãy thay tệp archive bị lỗi rồi chạy lại.")

    print("Đang giải nén VOC2012...")
    extract_archive(str(local_archive), str(data_root))

    if not drive_archive.exists():
        print("Lưu archive lên Drive để tái sử dụng...")
        shutil.copy2(local_archive, drive_archive)

print("Dữ liệu:", voc_root)

## 5. Kiểm tra split và tệp dữ liệu

Giữ nguyên split segmentation chính thức. Kiểm tra ID duy nhất, không trùng train/val,
và đủ ảnh, XML, mask. Các mask trong kiểm tra này không được đưa vào classifier.

In [ ]:
split_dir = voc_root / "ImageSets" / "Segmentation"
train_ids = (split_dir / "train.txt").read_text().split()
val_ids = (split_dir / "val.txt").read_text().split()

assert len(train_ids) == 1464 and len(val_ids) == 1449, "Sai split VOC2012 gốc."
assert len(set(train_ids)) == len(train_ids)
assert len(set(val_ids)) == len(val_ids)
assert set(train_ids).isdisjoint(val_ids), "Train và val bị trùng ID."

missing_files = [
    str(voc_root / folder / f"{image_id}{suffix}")
    for image_id in train_ids + val_ids
    for folder, suffix in [
        ("JPEGImages", ".jpg"),
        ("Annotations", ".xml"),
        ("SegmentationClass", ".png"),
    ]
    if not (voc_root / folder / f"{image_id}{suffix}").is_file()
]
assert not missing_files, f"Thiếu {len(missing_files)} tệp. Ví dụ: {missing_files[:3]}"
print(
    "Train:", len(train_ids),
    "| Val:", len(val_ids),
    "| Thiếu tệp:", len(missing_files),
)

## 6. Đọc nhãn cấp ảnh từ XML

Mỗi ảnh có vector 20 nhãn 0/1. Chỉ lấy tên lớp, giữ cả lớp xuất hiện ở đối tượng
được đánh dấu difficult; đây là quy ước nhãn cấp ảnh của thử nghiệm này.
Thứ tự lớp được lưu cùng checkpoint để tránh nhầm cột khi dự đoán.

In [ ]:
classes = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]

def build_label_table(image_ids):
    rows = []

    for image_id in image_ids:
        annotation = ET.parse(voc_root / "Annotations" / f"{image_id}.xml").getroot()
        names = {obj.findtext("name") for obj in annotation.findall("object")}

        if not names or not names.issubset(set(classes)):
            raise ValueError(f"Nhãn bất thường ở {image_id}: {names}")

        row = {"image_id": image_id}
        row.update({name: int(name in names) for name in classes})
        rows.append(row)

    return pd.DataFrame(rows, columns=["image_id", *classes])

df_train_labels = build_label_table(train_ids)
df_val_labels = build_label_table(val_ids)

for table in [df_train_labels, df_val_labels]:
    assert table["image_id"].is_unique
    assert table[classes].isin([0, 1]).all().all()
    assert (table[classes].sum(axis=1) > 0).all()

print("Bảng train:", df_train_labels.shape, "| Bảng val:", df_val_labels.shape)
display(df_train_labels.head())

## 7. Xem một ảnh, mask chuẩn và phân bố lớp

Mask chuẩn có 0 = nền, 1–20 = lớp đối tượng, 255 = ignore. Đổi mask sang RGB chỉ
để hiển thị; giữ mảng mã lớp khi cần đánh giá phân đoạn sau này.

In [ ]:
image_id = train_ids[0]

with Image.open(voc_root / "JPEGImages" / f"{image_id}.jpg") as source:
    sample_image = source.convert("RGB")

with Image.open(voc_root / "SegmentationClass" / f"{image_id}.png") as source:
    sample_mask = source.copy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(sample_image)
axes[0].set_title(f"Ảnh gốc: {image_id}")
axes[1].imshow(sample_mask.convert("RGB"))
axes[1].set_title("Mask chuẩn — dùng để xem/đánh giá")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

class_counts = df_train_labels[classes].sum().sort_values()
class_counts.plot.barh(figsize=(8, 6), xlabel="Số ảnh chứa lớp", title="Nhãn tập train")
plt.tight_layout()
plt.show()

## 8. Dataset và DataLoader

Dataset đọc ảnh từ tệp theo từng mẫu. Ảnh resize về cùng kích thước để gom batch.
ToTensor đưa ảnh về 0–1; Normalize sẽ được áp dụng đúng một lần trong train/eval.
Tạo cả train_loader và val_loader trước vòng huấn luyện.

In [ ]:
image_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
    ]
)

class VOCClassificationDataset(Dataset):
    def __init__(self, voc_root, labels_df, classes, transform):
        self.image_dir = voc_root / "JPEGImages"
        self.image_ids = labels_df["image_id"].tolist()
        self.targets = torch.tensor(labels_df[classes].to_numpy(dtype="float32"))
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_path = self.image_dir / f"{self.image_ids[index]}.jpg"

        with Image.open(image_path) as source:
            image = self.transform(source.convert("RGB"))

        return image, self.targets[index]

train_dataset = VOCClassificationDataset(
    voc_root,
    df_train_labels,
    classes,
    image_transform,
)

val_dataset = VOCClassificationDataset(
    voc_root,
    df_val_labels,
    classes,
    image_transform,
)

shuffle_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    generator=shuffle_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

images, targets = next(iter(train_loader))

assert images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
assert targets.shape == (images.size(0), len(classes))
print("Batch ảnh:", images.shape, "| Batch nhãn:", targets.shape)

## 9. ResNet pretrained, loss và optimizer

Fine-tuning toàn bộ model. Phần trích đặc trưng dùng trọng số ImageNet; lớp cuối 20 nhãn
được khởi tạo mới. BCEWithLogitsLoss nhận logits thô, không sigmoid trước loss.
Chạy lại cell này sẽ tạo model và optimizer mới từ đầu.

In [ ]:
if MODEL_NAME == "resnet18":
    weights = ResNet18_Weights.DEFAULT
    model = resnet18(weights=weights)
else:
    weights = ResNet50_Weights.DEFAULT
    model = resnet50(weights=weights)

model.fc = nn.Linear(model.fc.in_features, len(classes))

model = model.to(device)

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
normalize = transforms.Normalize(mean=mean, std=std)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Chạy kiểm tra đầu ra, không cập nhật trọng số ngoài vòng train chính.
model.eval()

with torch.no_grad():
    sample_logits = model(normalize(images).to(device))

assert sample_logits.shape == targets.shape
print("Model:", MODEL_NAME, "| Lớp cuối:", model.fc)
print("Đầu ra:", sample_logits.shape)
del sample_logits, images, targets

## 10. Huấn luyện và validation

Train bật gradient và cập nhật trọng số; validation dùng val_loader, tắt gradient.
Loss được cộng theo `loss.item() * batch_size` rồi chia tổng số ảnh.
Cell định nghĩa hàm chưa chạy huấn luyện.

In [ ]:
def run_epoch(loader, training):
    model.train(training)

    total_loss = 0.0
    total_images = 0
    progress = tqdm(loader, desc="Train" if training else "Validation")

    with torch.set_grad_enabled(training):
        for images, targets in progress:
            model_inputs = normalize(images).to(device)
            targets = targets.to(device)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(model_inputs)
            loss = criterion(logits, targets)

            if not torch.isfinite(loss).item():
                raise FloatingPointError("Loss không hữu hạn. Dừng để kiểm tra dữ liệu/mô hình.")

            if training:
                loss.backward()
                optimizer.step()

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_images += batch_size
            progress.set_postfix(avg_loss=f"{total_loss / total_images:.4f}")

    if total_images == 0:
        raise ValueError("Loader không có mẫu.")

    return total_loss / total_images

## 11. Chạy các epoch và lưu checkpoint lên Drive

Đây là cell bắt đầu huấn luyện. `best.pt` được chọn theo **val loss**, `last.pt` lưu
epoch cuối hoàn thành. `history.json` lưu đầy đủ lịch sử của lượt chạy, kể cả các epoch
sau checkpoint best. Epoch được đếm từ 1 của lượt chạy này.

In [ ]:
config = {
    "backbone": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "seed": SEED,
    "pretrained_weights": str(weights),
    "mean": mean,
    "std": std,
    "classes": list(classes),
    "train_images": len(train_ids),
    "val_images": len(val_ids),
    "train_ids_sha256": hashlib.sha256("\n".join(train_ids).encode()).hexdigest(),
    "val_ids_sha256": hashlib.sha256("\n".join(val_ids).encode()).hexdigest(),
    "torch_version": str(torch.__version__),
    "torchvision_version": str(torchvision.__version__),
}

(run_dir / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

history = []
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
        }
    )

    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss

    checkpoint = {
        "backbone": MODEL_NAME,
        "epoch_in_run": epoch,
        "classes": list(classes),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "best_val_loss": best_val_loss,
        "history": list(history),
        "config": config,
    }

    torch.save(checkpoint, run_dir / "last.pt")

    if improved:
        torch.save(checkpoint, run_dir / "best.pt")
        print("Đã lưu best.pt tốt hơn.")

    (run_dir / "history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8",
    )
    print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

del checkpoint

print("Kết quả đã lưu tại:", run_dir)

## 12. Vẽ loss và nạp model tốt nhất

Vẽ bằng history đầy đủ trong lượt chạy. Sau đó nạp best.pt để đánh giá và dự đoán.
Các cell còn lại phục vụ inference; nếu muốn tiếp tục train từ best, cần khôi phục cả
optimizer trong checkpoint tương ứng hoặc tạo optimizer mới.

In [ ]:
epochs = [row["epoch"] for row in history]
plt.figure(figsize=(7, 4))
plt.plot(epochs, [row["train_loss"] for row in history], marker="o", label="Train")
plt.plot(epochs, [row["val_loss"] for row in history], marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.xticks(epochs)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(run_dir / "loss_curve.png", dpi=150)
plt.show()

best_checkpoint = torch.load(
    run_dir / "best.pt",
    map_location="cpu",
    weights_only=True,
)
assert best_checkpoint["backbone"] == MODEL_NAME
assert best_checkpoint["classes"] == classes
model.load_state_dict(best_checkpoint["model_state_dict"])
model.to(device).eval()
best_epoch = best_checkpoint["epoch_in_run"]
recorded_best_loss = best_checkpoint["val_loss"]
del best_checkpoint

print("Best epoch:", best_epoch, "| Val loss đã lưu:", recorded_best_loss)

## 13. Đánh giá toàn bộ val: BCE loss và mAP phân loại ảnh

Thu thập điểm liên tục, không chuyển thành nhãn 0/1 trước khi tính AP. AP được tính bằng
scikit-learn và mAP là trung bình AP của 20 lớp, với quy ước nhãn XML đã nêu ở trên.
Đây là đánh giá phân loại cấp ảnh; chất lượng CAM và mIoU phân đoạn cần đánh giá riêng.

In [ ]:
all_targets, all_scores = [], []
total_val_loss = 0.0
total_val_images = 0

model.eval()

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Đánh giá best.pt"):
        logits = model(normalize(images).to(device))
        loss = criterion(logits, targets.to(device))
        batch_size = images.size(0)
        total_val_loss += loss.item() * batch_size
        total_val_images += batch_size
        all_targets.append(targets.numpy())
        all_scores.append(torch.sigmoid(logits).cpu().numpy())

y_true = np.concatenate(all_targets, axis=0)
y_score = np.concatenate(all_scores, axis=0)
positive_counts = y_true.sum(axis=0)

assert y_true.shape == (len(val_dataset), len(classes))
assert (positive_counts > 0).all(), "Có lớp không có ảnh dương, cần xem lại split."
ap_per_class = average_precision_score(y_true, y_score, average=None)
classification_map = float(ap_per_class.mean())

actual_val_loss = total_val_loss / total_val_images

if not math.isclose(actual_val_loss, recorded_best_loss, rel_tol=1e-4, abs_tol=1e-6):
    raise RuntimeError("Val loss tính lại không khớp checkpoint. Kiểm tra transform và split.")

ap_table = pd.DataFrame(
    {
        "class": classes,
        "AP (%)": ap_per_class * 100,
        "positive_images": positive_counts.astype(int),
    }
)
print(f"Val loss tính lại: {actual_val_loss:.6f}")
print(f"Classification mAP: {classification_map * 100:.2f}%")
display(ap_table.sort_values("AP (%)", ascending=False))

metrics = {
    "best_epoch": best_epoch,
    "val_loss": actual_val_loss,
    "classification_mAP": classification_map,
    "val_images": total_val_images,
    "per_class": ap_table.to_dict(orient="records"),
}
(run_dir / "metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

## 14. Xem dự đoán một ảnh validation

Đổi sample_index để xem ảnh khác. Ngưỡng 0.5 chỉ dùng để xem dự đoán thử;
mAP ở trên sử dụng điểm liên tục. Notebook kết thúc ở classifier, bước tiếp theo là tạo CAM.

In [ ]:
sample_index = min(5, len(val_dataset) - 1)
image, target = val_dataset[sample_index]

with torch.no_grad():
    scores = torch.sigmoid(model(normalize(image).unsqueeze(0).to(device)))[0].cpu()

true_labels = [name for i, name in enumerate(classes) if target[i].item() == 1]
predicted_labels = [name for i, name in enumerate(classes) if scores[i].item() >= 0.5]
plt.figure(figsize=(5, 5))
plt.imshow(image.permute(1, 2, 0))
plt.axis("off")
plt.tight_layout()
plt.savefig(run_dir / "sample_prediction.png", dpi=150)
plt.show()
print("Ảnh:", val_dataset.image_ids[sample_index])
print("Nhãn thật:", true_labels)
print("Dự đoán:", predicted_labels)
print("Checkpoint tốt nhất:", run_dir / "best.pt")

In [ ]:
sample_index = 5
class_name = "sheep"

image, target = val_dataset[sample_index]
class_index = classes.index(class_name)

assert target[class_index].item() == 1

model.eval()

feature_extractor = nn.Sequential(
    *list(model.children())[:-2]
).eval()

model_input = normalize(image).unsqueeze(0).to(device)

with torch.no_grad():
    feature_maps = feature_extractor(model_input)

    pooled_features = feature_maps.mean(dim=(2,3))
    logits = model.fc(pooled_features)

    score = torch.sigmoid(logits)[0, class_index].item()

print("Bản đồ đặc trưng:", feature_maps.shape)
print(f"Điểm dự đoán {class_name}: {score:.3f}")


In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    class_weights = model.fc.weight[class_index]

    raw_cam = (
        feature_maps[0] * class_weights[:, None, None]
    ).sum(dim = 0)

    cam = torch.relu(raw_cam)

    cam = F.interpolate(
        cam[None, None],
        size = image.shape[-2:],
        mode = "bilinear",
        align_corners=False
    )[0, 0]

    cam = cam / cam.max().clamp_min(1e-8)

heatmap = cam.cpu().numpy()
image_for_display = image.permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(image_for_display)
axes[0].set_title("Ảnh đầu vào")

axes[1].imshow(heatmap, cmap="jet", vmin=0, vmax=1)
axes[1].set_title(f"CAM: {class_name}")

axes[2].imshow(image_for_display)
axes[2].imshow(heatmap, cmap="jet", alpha=0.4, vmin=0, vmax=1)
axes[2].set_title("CAM chồng lên ảnh")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.5

# Vùng có mức kích hoạt đủ cao.
foreground = heatmap >= threshold

# 255: bỏ qua pixel này khi tính loss segmentation.
pseudo_mask = np.full(
    heatmap.shape,
    fill_value=255,
    dtype=np.uint8,
)

# Vector classification có index 0–19.
# Nhãn segmentation VOC tương ứng là 1–20.
pseudo_mask[foreground] = class_index + 1

print("Các giá trị trong mask:", np.unique(pseudo_mask))
print(f"Tỷ lệ pixel được gán nhãn: {foreground.mean():.2%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image_for_display)
axes[0].set_title("Ảnh đầu vào")

axes[1].imshow(foreground, cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"Vùng CAM ≥ {threshold}")

axes[2].imshow(image_for_display)
axes[2].imshow(
    np.ma.masked_where(~foreground, foreground),
    cmap="autumn",
    vmin=0,
    vmax=1,
    alpha=0.6,
)
axes[2].set_title("Vùng tạm gán nhãn sheep")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
image_id = val_dataset.image_ids[sample_index]

mask_path = voc_root / "SegmentationClass" / f"{image_id}.png"

with Image.open(mask_path) as source:
    # Giữ nguyên mã lớp, không convert sang RGB.
    gt_mask = np.array(source)

class_id = class_index + 1
gt_foreground = gt_mask == class_id

print("Ảnh:", image_id)
print("Kích thước mask thật:", gt_mask.shape)
print("Các nhãn:", np.unique(gt_mask))

In [ ]:
cam_original = F.interpolate(
    torch.from_numpy(heatmap)[None, None].float(),
    size=gt_mask.shape,
    mode="bilinear",
    align_corners=False,
)[0, 0].numpy()

eval_threshold = 0.5
pred_cam = cam_original >= eval_threshold

In [ ]:
def binary_iou(pred_mask, gt_mask, class_id):
    pred_mask = np.asarray(pred_mask, dtype=bool)

    if pred_mask.shape != gt_mask.shape:
        raise ValueError("Mask dự đoán và mask thật phải cùng kích thước.")

    valid = gt_mask != 255
    gt_foreground = gt_mask == class_id

    intersection = np.count_nonzero(
        pred_mask & gt_foreground & valid
    )

    union = np.count_nonzero(
        (pred_mask | gt_foreground) & valid
    )

    return intersection / union if union > 0 else float("nan")


iou_cam = binary_iou(pred_cam, gt_mask, class_id)

results = {
    "CAM": iou_cam,
}

print(f"IoU CAM — {class_name}: {iou_cam:.2%}")

In [ ]:
valid = gt_mask != 255

with Image.open(voc_root / "JPEGImages" / f"{image_id}.jpg") as source:
    original_image = np.array(source.convert("RGB"))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original_image)
axes[0].set_title("Ảnh gốc")

axes[1].imshow(
    np.ma.masked_where(~valid, gt_foreground),
    cmap="gray",
    vmin=0,
    vmax=1,
)
axes[1].set_title("Mask sheep thật")

axes[2].imshow(
    np.ma.masked_where(~valid, pred_cam),
    cmap="gray",
    vmin=0,
    vmax=1,
)
axes[2].set_title(f"CAM — IoU: {iou_cam:.2%}")

for ax in axes:
    ax.set_facecolor("lightgray")
    ax.axis("off")

plt.tight_layout()
plt.show()